In [19]:
import matplotlib
import platform
print ("Operating system: ", platform.system())
if "Linux" in platform.system():
    %matplotlib tk
else:
    %matplotlib qt
    
#
import matplotlib.pyplot as plt
%autosave 180
%load_ext autoreload
%autoreload 2
import numpy as np

#
import scipy
import os
import time

#
import pickle 


#
from calibration.CalibrationTools import CalibrationTools, get_binary_std_map, get_rois_stardist2d, get_img_std
from drift.drift import (make_template, compute_drift_multi_frames, correct_drift, 
                         correct_drift_single_frame, template_generation, 
                         plot_mean_vs_template, make_motion_template_and_correct_data)
from utils.utils import smooth_ca_time_series, compute_dff0



Operating system:  Linux


Autosaving every 180 seconds
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
#######################################################################
########### LOAD PRE-BMI DATA (e.g. 10-15mins recording) ##############
#######################################################################
fname = r'F:\bmi\DON10775\22-07-25\calibration\Image_001_001.raw'
#fname = '/media/cat/4TB/donato/DON-009460/22-06-21/calibration/Image_001_001.raw'
#fname = '/media/cat/4TB/donato/DON-008499/mouse2_calibration/Image_001_001.raw'
fname = '/media/cat/4TBSSD/donato/test/data/Image_001_001.raw'
#fname = '/home/cat/data/donato/bscope_tests/8499/data/Image_001_001.raw'
#fname = '/media/cat/4TB/donato/DON-10798/22-07-26/calibration/Image_001_001.raw'
fname = '/media/cat/4TB1/donato/DON-10795/22-07-30/calibration/Image_001_001.raw'
#fname = r'F:\bmi\DON10795\22-08-02\calibration\Image_001_001.raw'

# 
bmi_c = CalibrationTools(fname)

#
bmi_c.smooth_ca_time_series = smooth_ca_time_series
bmi_c.compute_dff0 = compute_dff0

#
bmi_c.subsample = 10 # for std computation downsample to every N'th frame; the more frames the better the rois;
                  #   TODO: use correlation instead?! might be much faster; it is quit fast in other implemenations


memmap :  (20000, 512, 512)


In [3]:
##################################################################
############### MOTION CORRECTION STEP ###########################
##################################################################
# 
if False:
    start = time.time()
    bmi_c = make_motion_template_and_correct_data(bmi_c)
    plot_mean_vs_template(bmi_c)
    print ("total processing time: ", time.time()-start, " sec")
else:
    print ("Skipping template makgin step: ")
    bmi_c.template = np.mean(bmi_c.data[:1000],axis=0)
    
print ("DONE...")

Skipping template makgin step: 
DONE...


In [4]:
###################################################################
################# GET STD MAP TO GET CELL FOOTPRINTS ##############
###################################################################
# 
start = time.time()

# Filter data, then make map 
if True:
    std_map = bmi_c.filter_data_make_std_map()

# # just make std, and divide by mean
# # DOESN"T SEEM TO WORK WELL in M1; 
# MIGHT TRY IN CA3 IMPLANTS
# else:
#     idx = np.arange(bmi_c.data.shape[0])[::bmi_c.subsample]
#     data_sparse = bmi_c.data[idx]
    
#     # compute std
#     std = np.std(data_sparse.astype('float32'), axis=0)
    
#     # compute mean
#     mean = np.mean(data_sparse.astype('float32'), axis=0)
    
#     #
#     std_map = (std/mean)*100
    
#     std_map = std

#
print ("total processing time: ", time.time()-start, " sec")
print ("...DONE...")


data into analysis:  (2000, 512, 512)
 gaussian filter width:  1 , order:  0
done filtering... (TO CHECK which axis are we filtering!!)
total processing time:  32.04638171195984  sec
...DONE...


In [5]:
#################################################################
########### MAKE BINARY MAP FOR CELL REGISTRATOIN ###############
#################################################################

# get binary map
vmax = 300
smin, smax = get_binary_std_map(std_map,vmax)


In [6]:
##################################################

bmi_c, img_std = get_img_std(smin, smax, std_map, bmi_c)


max proj values (vmin, vmax):  46.03524229074887 74.66960352422905


In [7]:
#########################################################
########### GENERATE CELL SEGMENTATION ##################
#########################################################
min_size_roi = 100  #<---- increase to exclude really small cells
max_size_roi = 800  #<----- decrease to exclude multi-cell artificats
bmi_c.rois, bmi_c.footprints = get_rois_stardist2d(img_std,
                                               min_size_roi,
                                               max_size_roi)

There are 4 registered models for 'StarDist2D':

Name                  Alias(es)
────                  ─────────
'2D_versatile_fluo'   'Versatile (fluorescent nuclei)'
'2D_versatile_he'     'Versatile (H&E nuclei)'
'2D_paper_dsb2018'    'DSB 2018 (from StarDist 2D paper)'
'2D_demo'             None
None
Found model '2D_versatile_fluo' for 'StarDist2D'.
Loading network weights from 'weights_best.h5'.
Loading thresholds from 'thresholds.json'.
Using default values: prob_thresh=0.479071, nms_thresh=0.3.
1/1 [==============================] - 1s 516ms/step


looping over cells: 100%|██████████| 95/95 [00:00<00:00, 1130.83it/s]


In [8]:
#########################################################
########### REORDER CELLS BY SNR OR F0 ##################
#########################################################
order_type = 'snr'  # 'snr' or 'f0'
bmi_c.reorder_cells_by_snr_or_f0(order_type)  # this function also coputes the snr / f0s of the cells


computing roi traces for SNR indexing: 100%|██████████| 2000/2000 [00:04<00:00, 459.33it/s]


In [9]:
#############################################################
############## VISUALIZE ALL CELLS AND TRACES ###############
#############################################################
#
bmi_c.scale=3  
# <--- decrease to see cell traces better
bmi_c.trace_subsample = 10       # Subsample the time series to go faster;

# visualize traces
bmi_c.visualize_traces_snr_order(std_map)

# 

memmap :  (20000, 512, 512)


In [14]:
###############################################################
########### SELECT ENSEMBEL CELLS AND VISUALIZE ###############
###############################################################

# save ensemble rois
bmi_c.ensemble1 = [0,10,20,21,11]
bmi_c.ensemble2 = [1,13,22,5,15]
both = np.hstack((bmi_c.ensemble1, bmi_c.ensemble2))
print ("all cells:", both)

#
bmi_c.show_traces_ids(both)


all cells: [ 0 10 20 21 11  1 13 22  5 15]


### COMPUTE THE MIN AND MAXES FOR THE SELECTED ENSMEMLES

Some important points:

1. For now we are working in pixel absolute values for each cell.  

2. A better option might be to find the maximum peak of a cell during a window and then save that value and normalize all future events by that value. (note: any online BMI filtering/chagnig of data will need to account for this).


In [15]:
# ######################################################################
# ########### RECOMPUTE TRACES WITH SINGLE FRAME PRECISION #############
# ######################################################################
bmi_c.trace_subsample = 1        # Subsample the time series to go faster;

# visualize traces
#bmi_c.compute_traces2(std_map, both)
bmi_c.compute_traces_ensembles(std_map)

print ("DONE...")

100%|██████████| 20000/20000 [00:10<00:00, 1892.09it/s]


DONE...


In [26]:
#############################################
########### RUN THRSHOLD SETTER #############
#############################################
# 
bmi_c.sample_rate = 30
bmi_c.post_reward_lockout = 10   # reward lockout in seconds
bmi_c.balance_ensemble_rewards_flag = False   #this makes sure that both ensembles elicit a similar number of random rewards
bmi_c.rois_smooth_window = 15     # of frames to use to smooth the realtime signal
bmi_c.smooth_diff_function_flag = True    # use a kernel window to smooth current value

#
bmi_c.reward_rate = 0.5

# find 30% reward threshold
# We have 3 options: 
#   1) rewarding  E1-E2 going above some threshold
#   2) rewarding E2-E1 going above some threhosld
#   3) rewarding both
# The challenge is mapping the ensemble states to tone/stimulus space

#gr.find_reward_thresholds_low_and_high()
bmi_c.find_reward_thresholds_high()  # this only rewards when sound passes specific level

# this option rewards both ensembles 
#normalize_peaks = True   # this flag normalizes the peaks to make sure one ensembel
#                         # doesn't completely dominate the other;
                          # TODO: make sure that this is implemented in the bmi online component also
#bmi_c.find_reward_thresholds_absolute(normalize_peaks)


#
print ("thresholds: ", bmi_c.high)

############################################## 
bmi_c.plot_rewarded_ensembles()


#
bmi_c.reward_rate_scaling_factor = 0.9

#
bmi_c.high = bmi_c.high*bmi_c.reward_rate_scaling_factor
print ("bmi_c.high: scaled by: ", bmi_c.reward_rate_scaling_factor, ", final value:  ", bmi_c.high)


COMPUTED # of roi traces:  43


100%|██████████| 19985/19985 [00:00<00:00, 55446.08it/s]


low, high:  -1.836873800313217 1.4685170573485118
nsec recording:  666 max # of random rewards (i.e. every 30sec)  22
 @30% reward:  11
updated rewards #:  12 -0.6255687287206987 0.5001205572824458
thresholds:  0.5001205572824458
bmi_c.high: scaled by:  0.9 , final value:   0.4501085015542012


In [60]:
#############################################
########### RUN THRSHOLD SETTER #############
#############################################

# save all data to disk
# also add the tone values here as well that will be used for the experiment
bmi_c.low_freq = 2000
bmi_c.high_freq = 18000

# 
ensemble1_footprints = []
ensemble1_contours = []
for k in bmi_c.ensemble1:
    
    # get footprints
    temp = bmi_c.footprints[k]
    temp1 = temp[0]
    temp2 = temp[1]
    temp = np.vstack((temp1,temp2))
    ensemble1_footprints.append(temp.T)
    
    # get contours
    ensemble1_contours.append(bmi_c.compute_contour_map(std_map, [k]))

# get ensembel 2 footprints/contours
ensemble2_footprints = []
ensemble2_contours = []
for k in bmi_c.ensemble2:
    # get footprints
    temp = bmi_c.footprints[k]
    temp1 = temp[0]
    temp2 = temp[1]
    temp = np.vstack((temp1,temp2))
    ensemble2_footprints.append(temp.T)
    
    # get contours
    ensemble2_contours.append(bmi_c.compute_contour_map(std_map, [k]))
    
# get ensemble f0 baselines
ensemble1_f0s = []
for k in bmi_c.ensemble1:
    # get footprints
    ensemble1_f0s.append(bmi_c.roi_f0s[k])

# get ensemble f0 baselines
ensemble2_f0s = []
for k in bmi_c.ensemble2:
    # get footprints
    ensemble2_f0s.append(bmi_c.roi_f0s[k])
    
# also grab contours of cells; both contains all cell ids
# contours = bmi_c.compute_contour_map(std_map, both)
# print ("len: ", len(contours))        

# also grab contours of cells; both contains all cell ids
contours_all_cells = bmi_c.compute_contour_map(std_map, np.arange(len(bmi_c.footprints)))
contours_all_cells = np.array(contours_all_cells, dtype=object)

# save individual pixels of each cell - currently implemented in BMI
fname_out = os.path.join(os.path.split(os.path.split(fname)[0])[0],
                        'rois_pixels_and_thresholds.npz')
np.savez(fname_out,
            
            #
            ensemble1_footprints = ensemble1_footprints,
            ensemble1_contours = ensemble1_contours,
            ensemble1_f0s = ensemble1_f0s,

            # 
            ensemble2_footprints = ensemble2_footprints,
            ensemble2_contours = ensemble2_contours,
            ensemble2_f0s = ensemble2_f0s,
         
            #
            reward_rate = bmi_c.reward_rate,
            reward_rate_scaling_factor = bmi_c.reward_rate_scaling_factor,
                  
            #
            contours_all_cells = contours_all_cells,
            cell_centres = np.int32(bmi_c.rois)[both],
            cell_ids = both,
            all_rois = np.int32(bmi_c.rois),
            low_threshold = bmi_c.low,
            high_threshold = bmi_c.high,
            low_freq = bmi_c.low_freq,
            high_freq = bmi_c.high_freq,
            all_roi_traces_submsampled = bmi_c.roi_traces,

            #
            sample_rate = bmi_c.sample_rate,
            post_reward_lockout = bmi_c.post_reward_lockout,
            balance_ensemble_rewards_flag = bmi_c.balance_ensemble_rewards_flag,
            rois_smooth_window = bmi_c.rois_smooth_window,
            smooth_diff_function_flag = bmi_c.smooth_diff_function_flag,
            calibration_template = bmi_c.template,
            footprints = bmi_c.footprints
             
        )

# also save the entire object as a pickle
file_pi = open(os.path.join(os.path.split(fname_out)[0], "bmi_c.obj"), 'wb') 
bmi_c.data=None
pickle.dump(bmi_c, file_pi)


In [52]:
data = np.load('/media/cat/4TB1/donato/DON-10795/22-07-30/rois_pixels_and_thresholds.npz',
               allow_pickle=True)

ensemble1_footprints = data['ensemble1_footprints']


In [56]:
print (ensemble1_footprints[1])

[[313 384]
 [313 385]
 [313 386]
 [313 387]
 [313 388]
 [314 382]
 [314 383]
 [314 384]
 [314 385]
 [314 386]
 [314 387]
 [314 388]
 [314 389]
 [314 390]
 [314 391]
 [315 381]
 [315 382]
 [315 383]
 [315 384]
 [315 385]
 [315 386]
 [315 387]
 [315 388]
 [315 389]
 [315 390]
 [315 391]
 [315 392]
 [315 393]
 [315 394]
 [316 381]
 [316 382]
 [316 383]
 [316 384]
 [316 385]
 [316 386]
 [316 387]
 [316 388]
 [316 389]
 [316 390]
 [316 391]
 [316 392]
 [316 393]
 [316 394]
 [316 395]
 [317 380]
 [317 381]
 [317 382]
 [317 383]
 [317 384]
 [317 385]
 [317 386]
 [317 387]
 [317 388]
 [317 389]
 [317 390]
 [317 391]
 [317 392]
 [317 393]
 [317 394]
 [317 395]
 [317 396]
 [318 379]
 [318 380]
 [318 381]
 [318 382]
 [318 383]
 [318 384]
 [318 385]
 [318 386]
 [318 387]
 [318 388]
 [318 389]
 [318 390]
 [318 391]
 [318 392]
 [318 393]
 [318 394]
 [318 395]
 [318 396]
 [318 397]
 [318 398]
 [319 379]
 [319 380]
 [319 381]
 [319 382]
 [319 383]
 [319 384]
 [319 385]
 [319 386]
 [319 387]
 [319 388]